# Pathgen agent training

Train a supervised classifier that estimates `P(struggle_on_next_question = 1)`. It is the prediction component of the agent; the question-selection policy can use its probabilities to compare eligible questions. Run the cells in order from the folder containing the agent CSVs and `bkt_parameters.json`.

The supervised data uses real Junyi encounters. The generation pipeline supplies behavior features from earlier answers, out-of-fold BKT and difficulty estimates for training learners, and frozen training-derived estimates for validation and test learners. The notebook checks that learner assignments are disjoint, keeps IDs and completion timestamps out of the model, and reserves the test set until after model selection, calibration, and threshold tuning.

[Scikit-learn guidance on preprocessing and leakage](https://scikit-learn.org/stable/common_pitfalls.html) · [Probability calibration](https://scikit-learn.org/stable/modules/calibration.html)

## 1. Set up the notebook and feature contract

In [1]:
from hashlib import sha256
from pathlib import Path
import json
import pickle
from time import perf_counter

from IPython.display import display
import numpy as np
import pandas as pd
import sklearn
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.frozen import FrozenEstimator
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             brier_score_loss, confusion_matrix, f1_score, log_loss,
                             precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
TARGET = "struggle_on_next_question"
SKILL = "candidate_skill_id"
TRACKING = ["user_id", "candidate_time_done"]
FEATURES = [
    SKILL, "candidate_difficulty", "last_correct", "last_attempt_count", "last_hint_used",
    "last_duration_sec", "recent_correct_rate", "recent_hint_rate", "recent_mean_attempts",
    "recent_median_duration_sec", "candidate_skill_prior_encounters", "candidate_skill_bkt_mastery",
]
ALL_COLUMNS = [*TRACKING, *FEATURES, TARGET]
TIMES = ["last_duration_sec", "recent_median_duration_sec"]
SPLIT_FILES = {split: ROOT / f"agent_{split}.csv" for split in ("training", "validation", "testing")}
QUOTAS = {"training": 10_000, "validation": 2_500, "testing": 3_000}
BKT_JSON = ROOT / "bkt_parameters.json"
MODEL_PATH = ROOT / "agent_model.pkl"
SEED = 42
THREADS = 4
ACCURACY_GOAL = .80
MIN_STRUGGLE_RECALL = .25

assert all(path.is_file() for path in [*SPLIT_FILES.values(), BKT_JSON])
with BKT_JSON.open(encoding="utf-8") as file:
    bkt_parameters = json.load(file)
assert bkt_parameters["model"] == "four_parameter_bkt_no_forgetting"
SKILLS = sorted(bkt_parameters["skills"])
assert len(SKILLS) == 10 and len(FEATURES) == 12 and not set(TRACKING + [TARGET]) & set(FEATURES)
print(f"scikit-learn {sklearn.__version__} · pandas {pd.__version__} · NumPy {np.__version__}")
print("Found the three CSVs. The test CSV has not been read.")

scikit-learn 1.9.1 · pandas 3.0.6 · NumPy 2.5.3
Found the three CSVs. The test CSV has not been read.


## 2. Verify training and validation, then separate validation students by purpose

Model selection uses approximately half the validation learners. A separate quarter fits a possible probability calibrator. The last quarter chooses whether to use calibration and sets the decision threshold. Splits use learner IDs; each learner stays in one group. The final test CSV remains unopened at this stage.

In [2]:
def student_hash(prefix, user):
    return int.from_bytes(sha256(f"{prefix}{user}".encode()).digest()[:8], "big")


def original_split(user):
    cutoff = (1 << 64) // 5
    if student_hash("42|", user) < cutoff:
        return "testing"
    if student_hash("42|bkt-validation|", user) < cutoff:
        return "validation"
    return "training"


def load_split(name):
    data = pd.read_csv(SPLIT_FILES[name], dtype={"user_id": str, SKILL: str})
    if list(data.columns) != ALL_COLUMNS:
        raise ValueError(f"Unexpected {name} schema.")
    counts = data[SKILL].value_counts()
    if set(counts.index) != set(SKILLS) or not counts.eq(QUOTAS[name]).all():
        raise ValueError(f"Unexpected {name} skill quotas.")
    if data.drop(columns=TIMES).isna().any().any() or data.duplicated(TRACKING).any():
        raise ValueError(f"Missing required values or duplicate encounters in {name}.")
    if not data[TARGET].isin([0, 1]).all() or data[TARGET].nunique() != 2:
        raise ValueError(f"Invalid {name} target.")
    for column in ("last_correct", "last_hint_used"):
        if not data[column].isin([0, 1]).all():
            raise ValueError(f"Invalid binary feature: {column}.")
    for column in ("candidate_difficulty", "recent_correct_rate", "recent_hint_rate", "candidate_skill_bkt_mastery"):
        if not data[column].between(0, 1).all():
            raise ValueError(f"Invalid probability or rate: {column}.")
    numeric = data[[column for column in FEATURES if column != SKILL]].to_numpy(dtype=float)
    if np.isinf(numeric).any() or (numeric < 0).any():
        raise ValueError(f"Invalid numeric features in {name}.")
    if any(original_split(user) != name for user in data["user_id"].unique()):
        raise ValueError(f"A student is in the wrong split: {name}.")
    return data


training = load_split("training")
validation = load_split("validation")
training_users = set(training["user_id"])
validation_users = set(validation["user_id"])
assert not training_users & validation_users
assignment = {u: student_hash("42|agent-validation-role|", u) % 4 for u in validation_users}
roles = validation["user_id"].map(assignment)
select_data = validation[roles.lt(2)]
calibrate_data = validation[roles.eq(2)]
threshold_data = validation[roles.eq(3)]
groups = {"training": training, "model selection": select_data,
          "calibration": calibrate_data, "threshold tuning": threshold_data}
for name, frame in groups.items():
    assert set(frame[SKILL]) == set(SKILLS) and frame[TARGET].nunique() == 2, name
display(pd.DataFrame([
    {"partition": name, "rows": len(frame), "students": frame["user_id"].nunique(),
     "struggle_rate": frame[TARGET].mean()}
    for name, frame in groups.items()
]).round(4))
print("Student assignments, feature ranges, duplicate keys, and quotas checked.")

Student assignments, feature ranges, duplicate keys, and quotas checked.


,partition,rows,students,struggle_rate
0,training,100000,10146,0.2071
1,model selection,12170,1263,0.2090
2,calibration,6623,653,0.2061
3,threshold tuning,6207,652,0.2054


## 3. Encode, normalize, and fit candidate models

The preprocessing pipeline one-hot encodes skill IDs, imputes missing numeric values using training medians, adds a missing indicator when present in training, and scales numeric features with a training-fitted standard scaler. Model fitting uses the training CSV only. Model selection uses the average of the ten skill-level log losses, giving each skill equal weight.

The candidate without the Junyi difficulty estimate checks how much that feature contributes. Model depth and leaf size are kept modest because difficulty is estimated from a small set of Junyi templates.

In [3]:
def pipeline(estimator, features):
    numeric = [column for column in features if column != SKILL]
    preprocessor = ColumnTransformer([
        ("skill_one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), [SKILL]),
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", StandardScaler()),
        ]), numeric),
    ], remainder="drop")
    return Pipeline([("preprocess", preprocessor), ("classifier", estimator)])


def macro_log_loss(data, probability):
    actual = data[TARGET].to_numpy()
    skill_names = data[SKILL].to_numpy()
    return float(np.mean([log_loss(actual[skill_names == skill], probability[skill_names == skill], labels=[0, 1])
                          for skill in SKILLS]))


def positive_probability(model, data, features):
    with threadpool_limits(limits=THREADS):
        probability = model.predict_proba(data[features])[:, 1]
    assert np.isfinite(probability).all() and np.all((probability >= 0) & (probability <= 1))
    return probability


def bkt_probability(data):
    guess = data[SKILL].map({s: bkt_parameters["skills"][s]["guess"] for s in SKILLS}).to_numpy()
    slip = data[SKILL].map({s: bkt_parameters["skills"][s]["slip"] for s in SKILLS}).to_numpy()
    mastery = data["candidate_skill_bkt_mastery"].to_numpy()
    return 1 - (mastery * (1 - slip) + (1 - mastery) * guess)


def boosted(leaves, rounds, min_leaf, regularization):
    return HistGradientBoostingClassifier(max_leaf_nodes=leaves, max_iter=rounds,
                                          min_samples_leaf=min_leaf, l2_regularization=regularization,
                                          learning_rate=.07, early_stopping=False, random_state=SEED)


no_difficulty = [f for f in FEATURES if f != "candidate_difficulty"]
candidates = [
    ("Logistic regression", LogisticRegression(C=.5, max_iter=500, random_state=SEED), FEATURES),
    ("Gradient boosting (15 leaves)", boosted(15, 140, 95, 1.5), FEATURES),
    ("Gradient boosting (31 leaves)", boosted(31, 220, 115, 2.5), FEATURES),
    ("Gradient boosting (31 leaves, no template difficulty)", boosted(31, 220, 115, 2.5), no_difficulty),
]
trained = {}
selection_records = []
for name, estimator, features in candidates:
    started = perf_counter()
    fitted = pipeline(estimator, features)
    with threadpool_limits(limits=THREADS):
        fitted.fit(training[features], training[TARGET])
    probability = positive_probability(fitted, select_data, features)
    score = macro_log_loss(select_data, probability)
    trained[name] = (fitted, list(features))
    selection_records.append({"model": name, "macro_log_loss": score,
                              "roc_auc": roc_auc_score(select_data[TARGET], probability)})
    print(f"{name}: macro log loss {score:.4f} ({perf_counter() - started:.1f}s)", flush=True)

selection_table = pd.DataFrame(selection_records).sort_values("macro_log_loss", ignore_index=True)
selected_name = selection_table.iloc[0]["model"]
selected_pipeline, selected_features = trained[selected_name]
print(f"Selected: {selected_name}")
display(selection_table.round(5))
print(f"BKT-only macro log loss on the same selection learners: {macro_log_loss(select_data, bkt_probability(select_data)):.4f}")

Logistic regression: macro log loss 0.4347 (0.6s)
Gradient boosting (15 leaves): macro log loss 0.4238 (0.9s)
Gradient boosting (31 leaves): macro log loss 0.4232 (1.3s)
Gradient boosting (31 leaves, no template difficulty): macro log loss 0.4274 (1.2s)
Selected: Gradient boosting (31 leaves)
BKT-only macro log loss on the same selection learners: 0.4436


,model,macro_log_loss,roc_auc
0,Gradient boosting (31 leaves),0.42315,0.77582
1,Gradient boosting (15 leaves),0.42383,0.77497
2,"Gradient boosting (31 leaves, no template diff...",0.42738,0.77095
3,Logistic regression,0.43469,0.76169


## 4. Calibrate probabilities and fix the decision threshold

Fit a sigmoid calibrator on its own validation learners while freezing the selected pipeline. Use it only if it improves equal-weight log loss on the threshold-tuning learners. The threshold is for binary reporting; Pathgen can rank candidates using the probability directly.

Among thresholds with at least 80% validation accuracy and at least 25% struggle recall, choose the one with the highest F1. If no threshold satisfies both constraints, choose the highest-F1 threshold and report that fallback. This rule also sets a comparable threshold for the BKT baseline.

In [4]:
calibrated = CalibratedClassifierCV(FrozenEstimator(selected_pipeline), method="sigmoid")
with threadpool_limits(limits=THREADS):
    calibrated.fit(calibrate_data[selected_features], calibrate_data[TARGET])
raw_probability = positive_probability(selected_pipeline, threshold_data, selected_features)
calibrated_probability = positive_probability(calibrated, threshold_data, selected_features)
raw_score = macro_log_loss(threshold_data, raw_probability)
calibrated_score = macro_log_loss(threshold_data, calibrated_probability)
use_calibration = calibrated_score < raw_score
model = calibrated if use_calibration else selected_pipeline
calibration_name = "sigmoid" if use_calibration else "none"
tuning_probability = calibrated_probability if use_calibration else raw_probability


def choose_threshold(outcomes, probability):
    rows = []
    for value in np.linspace(.05, .95, 181):
        predicted = probability >= value
        rows.append({"threshold": float(value), "accuracy": accuracy_score(outcomes, predicted),
                     "recall": recall_score(outcomes, predicted, zero_division=0),
                     "f1": f1_score(outcomes, predicted, zero_division=0)})
    grid = pd.DataFrame(rows)
    meets = grid[grid["accuracy"].ge(ACCURACY_GOAL) & grid["recall"].ge(MIN_STRUGGLE_RECALL)]
    pool = meets if not meets.empty else grid
    choice = pool.sort_values(["f1", "accuracy", "threshold"], ascending=[False, False, True]).iloc[0]
    return {**choice.to_dict(), "met_validation_constraints": not meets.empty}


agent_tuning = choose_threshold(threshold_data[TARGET], tuning_probability)
bkt_tuning = choose_threshold(threshold_data[TARGET], bkt_probability(threshold_data))
agent_threshold = agent_tuning["threshold"]
bkt_threshold = bkt_tuning["threshold"]
print(f"Threshold-set macro log loss: raw {raw_score:.4f}, sigmoid {calibrated_score:.4f}; selected calibration: {calibration_name}.")
display(pd.DataFrame({"Pathgen agent": agent_tuning, "BKT only": bkt_tuning}).T)
print("Model, calibration, and thresholds are fixed. The test CSV has not been read.")

Threshold-set macro log loss: raw 0.4112, sigmoid 0.4110; selected calibration: sigmoid.
Model, calibration, and thresholds are fixed. The test CSV has not been read.


,threshold,accuracy,recall,f1,met_validation_constraints
Pathgen agent,0.315,0.804737,0.507451,0.516361,True
BKT only,0.35,0.800548,0.418824,0.46314,True


## 5. Evaluate once on the held-out test students

Compare the frozen agent with BKT's predicted probability of an incorrect response, the skill-specific training struggle rate, and a rule that always predicts independent success. The BKT baseline is a close proxy for the struggle target because Junyi marks hint use as incorrect. Lower log loss and Brier score are better. Average precision summarizes the precision–recall curve.

In [5]:
test = load_split("testing")
test_users = set(test["user_id"])
assert not test_users & (training_users | validation_users)
actual = test[TARGET].to_numpy()
agent_probability = positive_probability(model, test, selected_features)
bkt_test_probability = bkt_probability(test)
skill_rates = training.groupby(SKILL)[TARGET].mean()
skill_probability = test[SKILL].map(skill_rates).to_numpy()


def metrics(outcomes, predicted, probability=None):
    result = {"accuracy": accuracy_score(outcomes, predicted),
              "balanced_accuracy": balanced_accuracy_score(outcomes, predicted),
              "precision": precision_score(outcomes, predicted, zero_division=0),
              "recall": recall_score(outcomes, predicted, zero_division=0),
              "f1": f1_score(outcomes, predicted, zero_division=0)}
    if probability is not None:
        result.update(log_loss=log_loss(outcomes, probability, labels=[0, 1]),
                      brier=brier_score_loss(outcomes, probability),
                      roc_auc=roc_auc_score(outcomes, probability),
                      average_precision=average_precision_score(outcomes, probability))
    return {key: float(value) for key, value in result.items()}


predicted = agent_probability >= agent_threshold
bkt_predicted = bkt_test_probability >= bkt_threshold
overall = {
    "Pathgen agent": metrics(actual, predicted, agent_probability),
    "BKT only": metrics(actual, bkt_predicted, bkt_test_probability),
    "Skill training rate": metrics(actual, skill_probability >= .5, skill_probability),
    "Always predict success": metrics(actual, np.zeros(len(actual), dtype=int)),
}
per_skill = []
for skill in SKILLS:
    mask = test[SKILL].eq(skill).to_numpy()
    y = actual[mask]
    row = metrics(y, predicted[mask], agent_probability[mask])
    per_skill.append({"skill": skill, "rows": int(mask.sum()), "struggle_rate": float(y.mean()),
                      **row, "bkt_log_loss": log_loss(y, bkt_test_probability[mask], labels=[0, 1])})
skill_table = pd.DataFrame(per_skill).set_index("skill")
confusion = confusion_matrix(actual, predicted, labels=[0, 1])
print(f"Held-out test: {len(test):,} rows from {len(test_users):,} students; struggle rate {actual.mean():.2%}.")
display(pd.DataFrame(overall).T.round(4))
print("Agent per skill, including BKT's log loss for comparison:")
display(skill_table[["rows", "struggle_rate", "accuracy", "recall", "f1", "log_loss", "bkt_log_loss"]].round(4))
display(pd.DataFrame(confusion, index=["Actual success", "Actual struggle"],
                     columns=["Predicted success", "Predicted struggle"]))
print(f"Equal-weight skill log loss: agent {skill_table['log_loss'].mean():.4f}; BKT {skill_table['bkt_log_loss'].mean():.4f}.")
print(f"Agent test accuracy: {overall['Pathgen agent']['accuracy']:.2%} at threshold {agent_threshold:.3f}.")

Held-out test: 30,000 rows from 3,169 students; struggle rate 20.68%.
Agent per skill, including BKT's log loss for comparison:
Equal-weight skill log loss: agent 0.4162; BKT 0.4342.
Agent test accuracy: 80.11% at threshold 0.315.


,accuracy,balanced_accuracy,precision,recall,f1,log_loss,brier,roc_auc,average_precision
Pathgen agent,0.8011,0.6921,0.5195,0.5064,0.5129,0.4162,0.1302,0.7781,0.5419
BKT only,0.8027,0.6556,0.5301,0.4048,0.4590,0.4342,0.1367,0.7539,0.4938
Skill training rate,0.7932,0.5000,0.0000,0.0000,0.0000,0.4755,0.1524,0.6790,0.3330
Always predict success,0.7932,0.5000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN


,rows,struggle_rate,accuracy,recall,f1,log_loss,bkt_log_loss
skill,,,,,,,
INT_ABS_COMPARE,3000,0.1220,0.8613,0.1284,0.1843,0.3402,0.3395
INT_ADD_NEG,3000,0.1663,0.8017,0.3647,0.3796,0.3981,0.4175
INT_CONTEXT,3000,0.2563,0.7437,0.5007,0.5003,0.4988,0.5291
INT_DIV_NEG,3000,0.1413,0.8453,0.1745,0.2418,0.3796,0.3894
INT_EXPR_NEG,3000,0.4270,0.7203,0.7963,0.7086,0.5150,0.5660
INT_MULT_NEG,3000,0.1087,0.8887,0.0920,0.1523,0.3196,0.3310
INT_NUMBER_LINE,3000,0.2487,0.7377,0.4678,0.4700,0.5049,0.5214
INT_OPPOSITE_POS4,3000,0.1583,0.8320,0.3284,0.3824,0.3946,0.4057
INT_ORDER_NEG,3000,0.0793,0.9197,0.1807,0.2630,0.2487,0.2612


,Predicted success,Predicted struggle
Actual success,20892,2905
Actual struggle,3062,3141


## 6. Save the fitted agent as a pickle

The pickle contains the fitted estimator, preprocessing, calibration choice, feature order, and selected threshold. It also records the input hashes and test results. Only load pickle files you trust. Teacher difficulty labels still need alignment with the Junyi template-difficulty values if the final model uses that feature.

```python
import pickle

with open("agent_model.pkl", "rb") as file:
    saved = pickle.load(file)
probability = saved["model"].predict_proba(candidates[saved["feature_columns"]])[:, 1]
flag_struggle = probability >= saved["decision_threshold"]
```

In [6]:
bundle = {
    "schema_version": 1,
    "model": model,
    "model_name": selected_name,
    "feature_columns": selected_features,
    "target": TARGET,
    "positive_class": 1,
    "decision_threshold": agent_threshold,
    "calibration": calibration_name,
    "sklearn_version": sklearn.__version__,
    "source_sha256": {name: sha256(path.read_bytes()).hexdigest() for name, path in SPLIT_FILES.items()},
    "bkt_parameters_sha256": sha256(BKT_JSON.read_bytes()).hexdigest(),
    "test_metrics": overall,
    "test_per_skill": per_skill,
    "test_confusion_matrix": confusion.tolist(),
    "test_rows": int(len(test)),
    "test_students": int(len(test_users)),
    "training_rows": int(len(training)),
    "validation_rows": int(len(validation)),
    "validation_threshold_metrics": {"agent": agent_tuning, "bkt": bkt_tuning},
}
temporary = MODEL_PATH.with_suffix(".pkl.tmp")
with temporary.open("wb") as file:
    pickle.dump(bundle, file, protocol=pickle.HIGHEST_PROTOCOL)
with temporary.open("rb") as file:
    reloaded = pickle.load(file)
sample = test.iloc[:64]
np.testing.assert_allclose(positive_probability(reloaded["model"], sample, reloaded["feature_columns"]),
                           agent_probability[:64], atol=1e-12, rtol=1e-12)
assert reloaded["decision_threshold"] == agent_threshold
temporary.replace(MODEL_PATH)
print(f"Saved and verified {MODEL_PATH.name} ({MODEL_PATH.stat().st_size:,} bytes).")

Saved and verified agent_model.pkl (794,398 bytes).
